In [12]:
# 임베딩, 저장 단계
# .env 읽기 -> 임베딩 모델 준비 -> DB 주소 꺼내기 -> PGVector로 DB연결 -> 문서 2개 준비 -> add_documents로 저장
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_postgres.vectorstores import PGVector
from langchain_core.documents import Document

load_dotenv()

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

connection_url = os.getenv("DATABASE_URL")

vector_store = PGVector(
    embeddings=embedding_model,
    collection_name="integrated_university_rules",
    connection=connection_url,
    use_jsonb=True
)

initial_documents = [
    Document(
        page_content="[도서관 규정] 학부생은 최대 5권까지 14일간 대출 가능합니다.",
        metadata={"source": "rules_library.txt", "category": "library", "year": 2026}
    ),
    Document(
        page_content="[장학금 규정] 직전 학기 평점 3.5 이상 시 장학금 대상입니다.",
        metadata={"source": "rules_scholarship.txt", "category": "scholarship", "year": 2026}
    ),
    Document(
        page_content="[기숙사 규정] 기숙사 입사 신청은 매 학기 평점 2.5 이상인 학생에 한해 가능합니다.",
        metadata={"source": "rules_dormitory.txt", "category": "dormitory", "year": 2026}
    ),
    Document(
        page_content="[수강신청 규정] 한 학기 최대 신청 가능 학점은 21학점이며, 초과 신청은 불가합니다.",
        metadata={"source": "rules_enrollment.txt", "category": "enrollment", "year": 2026}
    ),
    Document(
        page_content="[휴학 규정] 일반휴학은 통산 4학기까지 가능하며, 입학 후 1학기를 마쳐야 신청할 수 있습니다.",
        metadata={"source": "rules_leave.txt", "category": "leave_of_absence", "year": 2026}
    ),
    Document(
        page_content="[성적정정 규정] 성적 이의신청은 성적 발표일로부터 7일 이내에만 접수 가능합니다.",
        metadata={"source": "rules_grade.txt", "category": "grade", "year": 2026}
    ),
    Document(
        page_content="[졸업요건 규정] 졸업을 위해서는 전공 42학점, 교양 30학점을 포함해 총 130학점 이상을 이수해야 합니다.",
        metadata={"source": "rules_graduation.txt", "category": "graduation", "year": 2026}
    )

]

vector_store.add_documents(initial_documents) # 리스트를 받아서 DB에 저장하는 동작이다. 내부적으로 한개씩 꺼내서 저장하긴하지만, 그건 검색이아니라 저장이다. 순위 계싼 자체가 일어나지않음

['31b752f0-631a-4e02-a561-6c09b28e9789',
 '2760adea-720f-484b-aee9-29b07ce66fa3',
 '9de4aea8-11eb-42d8-8db5-b1255b973bde',
 '06048adb-a2b8-4505-af5f-c1c39d7c9bce',
 '44b59485-4a11-4170-8570-c0d9c08e5c58',
 '4436ecee-da82-4105-bbc6-51d5497ddeed',
 '628f01a6-28a4-48ee-8b7d-4428a7b11725']

In [13]:
# 리트리버로 검색하기
user_query = "학부생의 도서 대출 권수와 기간을 알려주세요."

retriever = vector_store.as_retriever()

searched_docs = retriever.invoke(user_query) # DB에 쿼리를 날려서 그 결과로 새로 만들어진 리스트이다. initial_documents를 그대로 재사용하는게 아니라, DB안에 있는 모든 문서와 쿼리 사이 거리를 계산한 뒤 그 결과를 바탕으로 새로 조합된 리스트이다. serched_docs에 도서관문서, 장학금문서가 다 들어가있는건 저 initial_documents 리스트를 그대로 갖고온게아니라 저거 두개뿐인데다가 k설정도 안해놔서 계산 결과가 둘다 상위권에 들어있기 때문이다. 우연한것.. 순위 계산은일어난건데 ..
# langchain/PGVector가 내부적으로 쿼리벡터로 변환해준다.

for doc in searched_docs:
    print(doc)
# 순위 계산과는 전혀 상관없는 출력 보기 좋게하는용도일뿐, 순위계산은 retriever.invoke(user_query)에서 끝나있는상태다. for문이 실행되는 시점엔 이미 결과가 확정된 리스트일뿐이다. 

page_content='[도서관 규정] 학부생은 최대 5권까지 14일간 대출 가능합니다.' metadata={'year': 2026, 'source': 'rules_library.txt', 'category': 'library'}
page_content='[기숙사 규정] 기숙사 입사 신청은 매 학기 평점 2.5 이상인 학생에 한해 가능합니다.' metadata={'year': 2026, 'source': 'rules_dormitory.txt', 'category': 'dormitory'}
page_content='[졸업요건 규정] 졸업을 위해서는 전공 42학점, 교양 30학점을 포함해 총 130학점 이상을 이수해야 합니다.' metadata={'year': 2026, 'source': 'rules_graduation.txt', 'category': 'graduation'}
page_content='[휴학 규정] 일반휴학은 통산 4학기까지 가능하며, 입학 후 1학기를 마쳐야 신청할 수 있습니다.' metadata={'year': 2026, 'source': 'rules_leave.txt', 'category': 'leave_of_absence'}


In [14]:
# 문서 리스트를 하나의 글로 이어 붙이기
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in searched_docs)

In [15]:
# 검색 + 생성을 하나의 체인으로 잇기
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

model = init_chat_model(
    model_provider="openai",
    model="gpt-6-luna",
    
)

parser = StrOutputParser()

prompt = ChatPromptTemplate.from_messages([
    ("system", """
    당신은 대학 학사 규정 안내 전문가입니다.
    제공된 [참고 문서]에만 근거해서 답변하고, 문서에 없는 내용은 '죄송하지만 알지 못합니다.'라고 답변하세요.

    [참고 문서]
    {context}
    """),
    ("user", "{question}")
])

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough() }
    | prompt
    | model
    | parser
)

response = chain.invoke("도서관 대출 권수와 기간 규정을 알려줘.")
print(response)

학부생은 책을 최대 5권까지 14일간 대출할 수 있습니다.


In [16]:
# vector_store.as_retriever() - 설정
"""
search_kwargs={
    "k": 3, Top-k설정, 유사도 기준 상위 3개 추출
    "filter": {...}, metadata 필터링 조건...
    "score_threshold": ..., -> 필수로 search_type="similiarity_score_threshold" 설정이 필수  # 임계값 설정
}
"""

'\nsearch_kwargs={\n    "k": 3, Top-k설정, 유사도 기준 상위 3개 추출\n    "filter": {...}, metadata 필터링 조건...\n    "score_threshold": ..., -> 필수로 search_type="similiarity_score_threshold" 설정이 필수  # 임계값 설정\n}\n'

In [17]:
# 리트리버를 거치지 않고 저장소(vector_store)에 직접 검색해서, 문서마다 코사인 거리 점수를 같이 받아 보는 셀이다.
# (저장소에서 직접 찾을 때, 점수를 보고 싶을 때)
query = "도서관 대출 권수와 기간 규정을 알려줘."

# 코사인 거리라서 0에 가까울수록 유사
searched_docs1 = vector_store.similarity_search_with_score(
    query=query
)

for doc, score in searched_docs1:
    print(f"score={score:.4f}, doc={doc}")

score=0.4497, doc=page_content='[도서관 규정] 학부생은 최대 5권까지 14일간 대출 가능합니다.' metadata={'year': 2026, 'source': 'rules_library.txt', 'category': 'library'}
score=0.7073, doc=page_content='[휴학 규정] 일반휴학은 통산 4학기까지 가능하며, 입학 후 1학기를 마쳐야 신청할 수 있습니다.' metadata={'year': 2026, 'source': 'rules_leave.txt', 'category': 'leave_of_absence'}
score=0.7119, doc=page_content='[기숙사 규정] 기숙사 입사 신청은 매 학기 평점 2.5 이상인 학생에 한해 가능합니다.' metadata={'year': 2026, 'source': 'rules_dormitory.txt', 'category': 'dormitory'}
score=0.7218, doc=page_content='[졸업요건 규정] 졸업을 위해서는 전공 42학점, 교양 30학점을 포함해 총 130학점 이상을 이수해야 합니다.' metadata={'year': 2026, 'source': 'rules_graduation.txt', 'category': 'graduation'}


In [18]:
# 거리 대신 관련도 점수로 바꿔 받고, 커트라인을 넘는 문서만 남기는 셀이다.
# 유사도임계치를 확인하는거지 거리가 아니다.!

# similarity_search_with_relevance_scores(..) : 유사도 임계치를 설정할 수 있음
SCORE_THRESHOLD = 0.7

# 1에 가까울수록 유사 0~1
searched_docs2 = vector_store.similarity_search_with_relevance_scores(
    query=query,
    score_threshold= 1 - SCORE_THRESHOLD # 관련도가 >= 0.3 (=거리 <= 0.7)
)

for doc, score in searched_docs2:
    print(f"score={score:.4f}, doc={doc}")
# print(searched_docs2)

score=0.5503, doc=page_content='[도서관 규정] 학부생은 최대 5권까지 14일간 대출 가능합니다.' metadata={'year': 2026, 'source': 'rules_library.txt', 'category': 'library'}


In [19]:
retriever2 = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 1 - SCORE_THRESHOLD}
)

searched_docs3 = retriever2.invoke(query)

searched_docs3

[Document(id='31b752f0-631a-4e02-a561-6c09b28e9789', metadata={'year': 2026, 'source': 'rules_library.txt', 'category': 'library'}, page_content='[도서관 규정] 학부생은 최대 5권까지 14일간 대출 가능합니다.')]